# Waypoint — Data Fetch Demo

Demonstrates fetching historical return data using the built-in catalog and custom instruments.

**Prerequisites**
- `uv sync --extra dev` completed
- `.env` file in the repo root with your API keys (see `.env.example`)

```
FRED_API_KEY=your_key_here
EODHD_API_KEY=your_key_here
```

In [1]:
from waypoint.catalog import (
    CPI_YOY,
    INTL_DEVELOPED,
    REAL_RATE_10Y,
    US_AGG_BONDS,
    US_LARGE_CAP,
    US_SMALL_CAP,
    US_TIPS,
)
from waypoint.data import fetch
from waypoint.instruments import Instrument

## 1. Inspect catalog instruments

Each `Instrument` carries its display name, vendor symbol, vendor, frequency, and classification metadata.

In [2]:
catalog_instruments = [
    US_LARGE_CAP,
    US_SMALL_CAP,
    INTL_DEVELOPED,
    US_AGG_BONDS,
    US_TIPS,
    REAL_RATE_10Y,
    CPI_YOY,
]

import polars as pl

pl.DataFrame([
    {
        "name": inst.name,
        "symbol": inst.symbol,
        "vendor": inst.vendor,
        "frequency": inst.frequency,
        "asset_class": inst.asset_class,
        "sub_asset_class": inst.sub_asset_class,
        "geography": inst.geography,
    }
    for inst in catalog_instruments
])

name,symbol,vendor,frequency,asset_class,sub_asset_class,geography
str,str,str,str,str,str,str
"""US Large Cap Equities""","""^SPX""","""yfinance""","""daily""","""Equities""","""Large Cap""","""US"""
"""US Small Cap Equities""","""^RUT""","""yfinance""","""daily""","""Equities""","""Small Cap""","""US"""
"""Intl Developed Equities""","""EFA""","""yfinance""","""daily""","""Equities""","""Developed""","""International"""
"""US Aggregate Bonds""","""AGG""","""yfinance""","""daily""","""Fixed Income""","""Aggregate""","""US"""
"""US TIPS""","""TIP""","""yfinance""","""daily""","""Fixed Income""","""Inflation-Linked""","""US"""
"""10Y Real Rate""","""DFII10""","""fred""","""daily""","""Macro""","""Real Rates""","""US"""
"""CPI YoY""","""CPIAUCSL""","""fred""","""monthly""","""Macro""","""Inflation""","""US"""


## 2. Define a custom instrument

Use `Instrument` directly for anything not in the catalog.

In [3]:
GOLD = Instrument(
    name="Gold",
    symbol="GLD",
    vendor="yfinance",
    frequency="daily",
    asset_class="Alternatives",
    sub_asset_class="Commodities",
    geography="Global",
)

GOLD

Instrument(name='Gold', symbol='GLD', vendor='yfinance', frequency='daily', asset_class='Alternatives', sub_asset_class='Commodities', geography='Global')

## 3. Fetch a single equity instrument

For `frequency="daily"` instruments, the date range is automatically snapped to full calendar months so resampling to monthly is always clean.

In [ ]:
spy = fetch(US_LARGE_CAP, start="2020-01-01", end="2024-12-31")

print(f"Name      : {spy.name}")
print(f"Ticker    : {spy.ticker}")
print(f"Frequency : {spy.frequency}")
print(f"Asset class: {spy.asset_class} / {spy.sub_asset_class} / {spy.geography}")
print(f"Periods   : {len(spy.returns):,} observations")
print(f"Date range: {spy.returns.equals(spy.returns)}")
spy.returns.head(10)

Name      : US Large Cap Equities
Ticker    : ^SPX
Frequency : daily
Asset class: Equities / Large Cap / US
Periods   : 1,257 observations
Date range: True


date,returns
date,f64
2020-01-03,-0.00706
2020-01-06,0.003533
2020-01-07,-0.002803
2020-01-08,0.004902
2020-01-09,0.006655
2020-01-10,-0.002855
2020-01-13,0.006976
2020-01-14,-0.001515
2020-01-15,0.00187


## 4. Fetch multiple instruments

Fetch a diversified set — equities, bonds, macro — and compare basic stats.

In [5]:
START = "2015-01-01"
END   = "2024-12-31"

assets = {
    inst.name: fetch(inst, start=START, end=END)
    for inst in [US_LARGE_CAP, US_SMALL_CAP, INTL_DEVELOPED, US_AGG_BONDS, GOLD]
}

for name, asset in assets.items():
    print(f"{name:<30} {len(asset.returns):>5} observations")

US Large Cap Equities           2515 observations
US Small Cap Equities           2515 observations
Intl Developed Equities         2515 observations
US Aggregate Bonds              2515 observations
Gold                            2515 observations


## 5. Fetch a FRED macro series

Requires `FRED_API_KEY` in `.env`. Monthly CPI data.

In [6]:
cpi = fetch(CPI_YOY, start="2015-01-01", end="2024-12-31")

print(f"Name      : {cpi.name}")
print(f"Frequency : {cpi.frequency}")
print(f"Periods   : {len(cpi.returns)} observations")
cpi.returns.tail(12)

Name      : CPI YoY
Frequency : monthly
Periods   : 119 observations


date,returns
date,f64
2024-01-01,0.0031
2024-02-01,0.004098
2024-03-01,0.004431
2024-04-01,0.002171
2024-05-01,0.000486
…,…
2024-08-01,0.001572
2024-09-01,0.002133
2024-10-01,0.002856


## 6. Quick return statistics

Show annualized mean and std for each fetched daily asset.

In [7]:
import numpy as np

rows = []
for name, asset in assets.items():
    ppy = asset.periods_per_year
    ann_return = float((1 + asset.returns).product() ** (ppy / len(asset.returns)) - 1)
    ann_vol    = float(asset.returns.std() * np.sqrt(ppy))
    rows.append({
        "instrument": name,
        "observations": len(asset.returns),
        "ann_return_%": round(ann_return * 100, 2),
        "ann_vol_%":    round(ann_vol    * 100, 2),
        "sharpe (rf=0)": round(ann_return / ann_vol, 2) if ann_vol else None,
    })

pl.DataFrame(rows)

TypeError: unsupported operand type(s) for ** or pow(): 'DataFrame' and 'float'

## 7. Force-refresh cached data

Use `force_refresh=True` to bypass the cache and re-fetch from the vendor (e.g. after a corporate action adjustment is published).

In [ ]:
spy_fresh = fetch(US_LARGE_CAP, start="2024-01-01", end="2024-12-31", force_refresh=True)
print(f"Re-fetched {len(spy_fresh.returns)} observations for {spy_fresh.name}")

Re-fetched 251 observations for US Large Cap Equities
